# 02 — Your First Agent
### One agent. Four tools. One transit classification.

---

**Format:** Narrated demo — pre-run outputs. Follow along or press Run All.

**The story:** Module 01 built a pipeline that loads, validates, engineers features, and detects anomalies in a TESS light curve. That pipeline is a black box to any LLM — it's just Python functions. This module changes that. We register those functions as **agent tools**, give a LangGraph ReAct agent access to them, and watch it reason through the same pipeline it would otherwise never know about.

```
Module 01                         Module 02
──────────────────────────────    ──────────────────────────────────────
load_lightcurve()      function → @tool  ← LLM reads the docstring
validate_lightcurve()  function → @tool  ← LLM decides when to call it
run_feature_anomaly_pipeline()  → @tool  ← LLM interprets the output
build_agent_context()  function → @tool  ← LLM assembles the summary
```

The pipeline doesn't change. It gets a smarter driver.

**Default inference target:** AI Navigator running locally at `http://localhost:8080/v1` — no API key needed. Override with any OpenAI-compatible endpoint via env vars.

---
## Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

# ── Path resolution ───────────────────────────────────────────────────────────
# Works whether you open this notebook from 02-your-first-agent/ or the repo root.
NOTEBOOK_DIR = Path(".").resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "02-your-first-agent" else NOTEBOOK_DIR

sys.path.insert(0, str(ROOT / "02-your-first-agent"))
sys.path.insert(0, str(ROOT / "01-data-sources"))

# ── Data ──────────────────────────────────────────────────────────────────────
# wasp18b_lightcurve.csv lives in the polars_demo git submodule.
# If you cloned without --recurse-submodules, run:
#   git submodule update --init --recursive
DATA_PATH = ROOT / "01-data-sources" / "wasp18b_lightcurve.csv"
if not DATA_PATH.exists():
    DATA_PATH = ROOT / "100-example-applications" / "polars_demo" / "wasp18b_lightcurve.csv"

if not DATA_PATH.exists():
    polars_demo = ROOT / "100-example-applications" / "polars_demo"
    if polars_demo.exists() and not any(polars_demo.iterdir()):
        raise FileNotFoundError(
            "polars_demo submodule is empty.\n"
            "Run from the repo root:\n"
            "  git submodule update --init --recursive\n"
            "Then re-open this notebook."
        )
    raise FileNotFoundError(
        "wasp18b_lightcurve.csv not found.\n"
        "Most likely fix:\n"
        "  git submodule update --init --recursive\n"
        "Or fetch directly:\n"
        "  python3 100-example-applications/polars_demo/fetch_data.py"
    )

print(f"✓  data   : {DATA_PATH}")
print(f"✓  python : {sys.version.split()[0]}")

---
## Part 1 — The pipeline as plain functions

The functions from `ingestion.py` work exactly as before. No change needed to use them from an agent. This cell is the "before" — the pipeline running without any LLM involved.

In [ ]:
from ingestion import SCHEMA, load_lightcurve, validate_lightcurve
from agent_tools import run_feature_anomaly_pipeline

# These are the same calls Module 01 made — nothing has changed
df     = load_lightcurve(DATA_PATH, SCHEMA)
report = validate_lightcurve(df)
result = run_feature_anomaly_pipeline(df)

print("ValidationReport:")
print(f"  nulls            : {report.nulls}")
print(f"  phase_range      : {report.phase_range}")
print(f"  flux_std         : {report.flux_std:.8f}")
print(f"  duplicate_phases : {report.duplicate_phases}")

tw = result['transit_window']
td = result['transit_depth']
print("\nAnomaly detection:")
print(f"  n_anomalous_points : {tw['n_anomalous_points']}")
print(f"  transit_depth_pct  : {td*100:.6f}" if td else "  transit_depth_pct  : None")
print(f"  transit_start      : {tw['transit_start']:+.6f}" if tw['transit_start'] else "  transit_start      : None")
print(f"  transit_end        : {tw['transit_end']:+.6f}" if tw['transit_end'] else "  transit_end        : None")

ValidationReport:
  nulls            : {'PHASE': 0, 'LC_DETREND': 0, 'MODEL_INIT': 0}
  phase_range      : (-0.4999767..., 0.4999547...)
  flux_std         : 0.00031482
  duplicate_phases : 0

Anomaly detection:
  n_anomalous_points : 91
  transit_depth_pct  : 1.014200
  transit_start      : -0.042318
  transit_end        :  0.042104


---
## Part 2 — Registering tools: the `@tool` decorator

The `@tool` decorator does one thing: it makes a Python function visible to a LangGraph agent. The agent reads the **docstring** to decide when to call the function. The **type hints** tell it what to pass. The **return value** is what the agent reasons about.

Write docstrings for the model, not a human reader. The agent is the consumer.

In [ ]:
from langchain_core.tools import tool
from agent_tools import build_agent_context, load_lightcurve, run_feature_anomaly_pipeline, validate_lightcurve, SCHEMA

@tool
def load_lightcurve_tool(filepath: str) -> str:
    """Load a WASP-18 lightcurve CSV and confirm row count.
    Always call this first before validation or analysis."""
    df = load_lightcurve(Path(filepath), SCHEMA)
    return f"Loaded {len(df)} rows from {Path(filepath).name}."

@tool
def validate_lightcurve_tool(filepath: str) -> str:
    """Validate a WASP-18 lightcurve CSV and return a structured JSON quality report.
    Checks for nulls, phase range, flux statistics, and schema conformance.
    Call this after loading to confirm data quality before anomaly detection."""
    df = load_lightcurve(Path(filepath), SCHEMA)
    return validate_lightcurve(df).model_dump_json(indent=2)

@tool
def feature_anomaly_tool(filepath: str) -> str:
    """Run feature engineering and IsolationForest anomaly detection on a lightcurve.
    Returns transit window boundaries, transit depth, and anomalous point count.
    Call this after validation to identify the planetary transit signal."""
    df = load_lightcurve(Path(filepath), SCHEMA)
    p = run_feature_anomaly_pipeline(df)
    return json.dumps({"transit_window": p["transit_window"], "transit_depth": p["transit_depth"]}, indent=2)

@tool
def build_context_tool(filepath: str) -> str:
    """Build the complete structured agent context for a lightcurve file.
    Combines validation report, feature summary, and anomaly results.
    Use this to assemble a final summary for the user."""
    return json.dumps(build_agent_context(filepath), indent=2)

TOOLS = [load_lightcurve_tool, validate_lightcurve_tool, feature_anomaly_tool, build_context_tool]

print("Registered tools:")
for t in TOOLS:
    first_line = t.description.split("\n")[0]
    print(f"  {t.name:<26}— {first_line}")

print("\nThe agent reads each docstring to decide when to call each tool.")
print("Type hints tell it what arguments to pass.")
print("Return values are what it reasons about.")

Registered tools:
  load_lightcurve_tool      — Load a WASP-18 lightcurve CSV and confirm row count.
  validate_lightcurve_tool  — Validate a WASP-18 lightcurve CSV and return a structured JSON quality report.
  feature_anomaly_tool      — Run feature engineering and IsolationForest anomaly detection on a lightcurve.
  build_context_tool        — Build the complete structured agent context for a lightcurve file.

The agent reads each docstring to decide when to call each tool.
Type hints tell it what arguments to pass.
Return values are what it reasons about.


---
## Part 3 — Building the agent

`create_react_agent` wires the LLM and tools into a ReAct loop: **Re**ason → **Act** (call a tool) → observe the result → reason again. It runs until the agent decides it has enough information to answer.

The `ChatOpenAI` client points at whatever `INFERENCE_BASE_URL` is set to — AI Navigator, vLLM, Anthropic, anything OpenAI-compatible. The agent code is identical for all of them.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

BASE_URL = os.environ.get("INFERENCE_BASE_URL", "http://localhost:8080/v1")
API_KEY  = os.environ.get("INFERENCE_API_KEY",  "not-needed")
MODEL    = os.environ.get("INFERENCE_MODEL",    "default")

print(f"Inference target : {BASE_URL}")
print(f"Model            : {MODEL}")

llm   = ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=API_KEY, temperature=0)
agent = create_react_agent(llm, TOOLS)

print(f"\n✓  Agent built — {len(TOOLS)} tools registered")
print("\nTo use a different inference endpoint:")
print("  AI Navigator (local):  INFERENCE_BASE_URL=http://localhost:8080/v1")
print("  Anthropic:             INFERENCE_BASE_URL=https://api.anthropic.com/v1")
print("                         INFERENCE_MODEL=claude-haiku-4-5-20251001")
print("  vLLM (self-hosted):    INFERENCE_BASE_URL=http://your-server:8000/v1")
print("                         INFERENCE_MODEL=mistralai/Mistral-7B-Instruct-v0.3")

Inference target : http://localhost:8080/v1
Model            : default

✓  Agent built — 4 tools registered

To use a different inference endpoint:
  AI Navigator (local):  INFERENCE_BASE_URL=http://localhost:8080/v1
  Anthropic:             INFERENCE_BASE_URL=https://api.anthropic.com/v1
                         INFERENCE_MODEL=claude-haiku-4-5-20251001
  vLLM (self-hosted):    INFERENCE_BASE_URL=http://your-server:8000/v1
                         INFERENCE_MODEL=mistralai/Mistral-7B-Instruct-v0.3


---
## Part 4 — Run the agent

The prompt tells the agent what file to work with and what order to call the tools. Watch the reasoning trace — each `[ai] → tool_name(args)` line is the agent deciding, in real time, what to do next.

In [ ]:
PROMPT = (
    f"You are an intelligent data analyst. The lightcurve file is at: {DATA_PATH}\n\n"
    "Use the available tools in this order:\n"
    "1. Load the WASP-18 lightcurve\n"
    "2. Validate its data quality\n"
    "3. Run anomaly detection\n"
    "4. Summarize the transit signal in plain language — depth, window, and "
    "what it tells us about the planet"
)

result = agent.invoke({"messages": [{"role": "user", "content": PROMPT}]})

# Print the reasoning trace — every tool call and intermediate response
print("── Agent reasoning trace ─────────────────────────────────────────\n")
for msg in result["messages"][:-1]:
    role       = getattr(msg, "type", type(msg).__name__)
    content    = getattr(msg, "content", "")
    tool_calls = getattr(msg, "tool_calls", [])
    if tool_calls:
        for tc in tool_calls:
            print(f"  [{role}] → {tc['name']}({tc['args']})")
    elif content:
        display = content if len(content) < 300 else content[:300] + "…"
        print(f"  [{role}] {display}")

── Agent reasoning trace ─────────────────────────────────────────

  [ai] → load_lightcurve_tool({'filepath': '.../wasp18b_lightcurve.csv'})
  [tool] Loaded 1800 rows from wasp18b_lightcurve.csv.
  [ai] → validate_lightcurve_tool({'filepath': '.../wasp18b_lightcurve.csv'})
  [tool] {"nulls": {"PHASE": 0, "LC_DETREND": 0, "MODEL_INIT": 0}, "flux_std": 0.00031482...}
  [ai] → feature_anomaly_tool({'filepath': '.../wasp18b_lightcurve.csv'})
  [tool] {"transit_window": {"transit_start": -0.042318, "transit_end": 0.042104...}}
  [ai] → build_context_tool({'filepath': '.../wasp18b_lightcurve.csv'})
  [tool] {"dataset": "wasp18b_lightcurve.csv", "anomaly_detection": {"transit_depth_pct": 1.0142...}}


In [ ]:
print("── Final answer ──────────────────────────────────────────────────\n")
print(result["messages"][-1].content)

── Final answer ──────────────────────────────────────────────────

The WASP-18 b light curve is clean (0 nulls, 1800 rows) with a clear
periodic transit signal. IsolationForest flagged 91 anomalous points
clustered between phase -0.042 and +0.042 days — a ~2-hour window
consistent with a hot Jupiter crossing the stellar disk.

Transit depth: 1.014% — the star dims by about 1% each orbit.
At a 22-hour period, this planet completes more orbits in a month
than Earth does in a year. The tidal forces at that distance are
slowly heating and stretching WASP-18 b.


---
## Part 5 — What just happened

```
User prompt
    │
    ▼
LLM reads tool docstrings → decides to call load_lightcurve_tool
    │
    ▼  "Loaded 1800 rows"
LLM decides to call validate_lightcurve_tool
    │
    ▼  {"nulls": {"PHASE": 0, ...}, "flux_std": 0.00031482}
LLM decides to call feature_anomaly_tool
    │
    ▼  {"transit_start": -0.042, "transit_depth": 0.01014}
LLM decides to call build_context_tool
    │
    ▼  full JSON context
LLM writes the final answer in plain language
```

The pipeline code is the same as Module 01. The LLM drove the sequence, interpreted the outputs, and produced a natural language summary — without knowing anything about Polars, IsolationForest, or light curve physics.

**What Module 03 does with this:** wraps this pattern in a Metaflow `FlowSpec` with `@conda` per step, adds a second `AnalysisAgent`, runs `foreach` parallelism across 50 targets, and deploys the whole thing to Outerbounds or a Linux box without changing a line of agent code.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║  🛸  WASP-18 b  ·  MODULE 02 COMPLETE                       ║
║                                                              ║
║  Your First Agent                                            ║
║  One agent. Four tools. One transit classification.          ║
║                                                              ║
║  Show this screen at the Anaconda booth to claim your prize. ║
║  🐍  PyCon US 2026  ·  Long Beach                           ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║  🛸  WASP-18 b  ·  MODULE 02 COMPLETE                       ║
║                                                              ║
║  Your First Agent                                            ║
║  One agent. Four tools. One transit classification.          ║
║                                                              ║
║  Show this screen at the Anaconda booth to claim your prize. ║
║  🐍  PyCon US 2026  ·  Long Beach                           ║
╚══════════════════════════════════════════════════════════════╝
